In [1]:
# from google.colab import drive
from pathlib import Path
import sys
# !pip install datasets transformers huggingface_hub
from datasets import load_dataset

# drive.mount('/content/drive/')
# strPath = "drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models"
# sys.path.insert(0,"drive/MyDrive/Spurious_Correlations/Identifying-and-Mitigating-Spurious-Correlations-for-Improving-Robustness-in-NLP-Models")
# filePath = Path(strPath)
# %cd $filePath
import torch
# !pip install captum

import numpy as np
from transformers import AutoTokenizer,AutoModelForSequenceClassification

from utils import data_structure

In [2]:
# attention_sst_sst = np.load("models/distilbert_yelp_base_uncased/cls_attention_weights.npy") # old attention weights
attention_sst_sst = np.load("models/distilbert_yelp_base_uncased/cls_attention_weights.npy") 
preds = np.load("models/distilbert_yelp_base_uncased/predictions.npy")
df = data_structure.get_yelp()
tokenizer = AutoTokenizer.from_pretrained("models/distilbert_yelp_base_uncased/") # make sure that the specified file has a config.json file in it
vocab_filename = "models/distilbert_yelp_base_uncased/vocab.txt"

  0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
idx = 2017
print(df.text[idx])
print(df.label[idx])
sentence = df.text[idx]
attr = attention_sst_sst[idx,:]
tokenized_sentence = tokenized_sentence=tokenizer.encode(sentence,add_special_tokens=False,padding=True,truncation=True,return_tensors='pt')
curr_pred = preds[idx]
label_pred=np.argmax(curr_pred)
print(attr)
print(curr_pred)
print(label_pred)

Very dirty!!!!\n\nAs soon as you walk in to this place you are hit with a terrible smell. You can tell that it is older so there is only so much that can be done. The laser tag itself was pretty decent but everything else wasnt great. The arcade is small and the food is terrible. Bathrooms were clean enough for me to use but all around not an impressive place.
0
[0.02075376 0.03030669 0.05062732 0.00810302 0.00704242 0.00647536
 0.00557386 0.00282366 0.00281133 0.00117941 0.01067871 0.00824764
 0.00320396 0.00796148 0.00321723 0.0033557  0.00445997 0.01582341
 0.01235488 0.01579569 0.01238247 0.01498636 0.01197122 0.01179307
 0.04283199 0.02357453 0.02454162 0.00282862 0.00218289 0.00231878
 0.00276486 0.00363762 0.00258482 0.00323034 0.00397409 0.00297595
 0.00504737 0.00903079 0.00295352 0.00171715 0.00187063 0.00170202
 0.00169469 0.00166822 0.03427158 0.00682799 0.0036439  0.00230206
 0.00201131 0.0108907  0.0038285  0.00337144 0.00659744 0.01480383
 0.00869184 0.03949232 0.0403515

In [4]:
temp=attr
if label_pred == 0:
  temp = temp * -1

for i in range(len(temp)):
    if temp[i]>0 or temp[i]<0:
        temp[i]=temp[i]*100
        

In [5]:
# Visualize the attribution for the attribution
from captum.attr import visualization as viz

indices = tokenized_sentence[0].detach().tolist()
all_tokens = tokenizer.convert_ids_to_tokens(indices)
delta=torch.Tensor([0])
confidence=np.max(curr_pred)
ground_label=0
last_arg=0

print(confidence)

attributions_start_sum = attr[0]

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        temp,
                        1, 
                        label_pred,
                        ground_label,
                        last_arg,
                        temp.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])

3.9418924
 Visualizations For Start Position 


In [6]:
idx = 2024
print(df.text[idx])
print(df.label[idx])
sentence = df.text[idx]
attr = attention_sst_sst[idx,:]
tokenized_sentence = tokenized_sentence=tokenizer.encode(sentence,add_special_tokens=False,padding=True,truncation=True,return_tensors='pt')
curr_pred = preds[idx]
label_pred=np.argmax(curr_pred)
# print(attr)
# print(curr_pred)
print(label_pred)

Yum. Baja fresh is the place to go if you want something a step up from Chipotle in variety and warmth as far as the food goes. \nIt's not finger licking if I don't have this I\""m gonna die good, but the food is made fresh, and the customer service is always excellent\nHighlight: the plethora of salsa choices as well as fresh cut limes for your meal & if you so choose your coke like beverage. :)
1
1


In [7]:
def combine_tokens(tokens, scores=None):
    if scores is None:
        token_stack = []

        if len(tokens) > 128:
            sep = tokens[-1]
            tokens = tokens[:127]
            tokens.append(sep)

        for idx, token in enumerate(tokens):

            if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif len(token_stack) > 0 and "'" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token
            elif "#" in token:
                pre_token = token_stack[-1]
                new_token = pre_token + token
                token_stack[-1] = new_token.replace("#", "")
            else:
                token_stack.append(token)
                
        return token_stack

    token_stack = []
    score_stack = []
    
    if len(tokens) > 128:
        sep = tokens[-1]
        tokens = tokens[:127]
        tokens.append(sep)
        
    for idx, token in enumerate(tokens):
                
        if len(token_stack) > 0 and token_stack[-1] == "'" and token == 't':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '##s' and token_stack[-1] == "her":
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '.' and token_stack[-1] in ['mr', 'mrs', 'ms']:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and token_stack[-1][-1] == '-':
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif len(token_stack) > 0 and "'" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token
            score_stack[-1] = new_score
        elif "#" in token:
            pre_token = token_stack[-1]
            pre_score = score_stack[-1]
            new_token = pre_token + token
            new_score = pre_score + scores[idx]
            token_stack[-1] = new_token.replace("#", "")
            score_stack[-1] = new_score
        else:
            token_stack.append(token)
            score_stack.append(scores[idx])
    assert len(token_stack) == len(score_stack)
    return token_stack, score_stack


In [8]:
word2id = dict()
id2word = dict()
with open(vocab_filename, "r") as f:
    lines = f.readlines()
for idx, line in enumerate(lines):
    word2id[line.strip()] = idx
    id2word[idx] = line.strip()

tokens = [id2word[item] for item in tokenizer(sentence)["input_ids"]] 
print(tokens)
tokens,new_attr = combine_tokens(tokens,attr)
print(tokens)
print(new_attr)



['[CLS]', 'yu', '##m', '.', 'baja', 'fresh', 'is', 'the', 'place', 'to', 'go', 'if', 'you', 'want', 'something', 'a', 'step', 'up', 'from', 'chip', '##ot', '##le', 'in', 'variety', 'and', 'warmth', 'as', 'far', 'as', 'the', 'food', 'goes', '.', '\\', 'ni', '##t', "'", 's', 'not', 'finger', 'licking', 'if', 'i', 'don', "'", 't', 'have', 'this', 'i', '\\', '"', '"', 'm', 'gonna', 'die', 'good', ',', 'but', 'the', 'food', 'is', 'made', 'fresh', ',', 'and', 'the', 'customer', 'service', 'is', 'always', 'excellent', '\\', 'nh', '##igh', '##light', ':', 'the', 'pl', '##eth', '##ora', 'of', 'salsa', 'choices', 'as', 'well', 'as', 'fresh', 'cut', 'lime', '##s', 'for', 'your', 'meal', '&', 'if', 'you', 'so', 'choose', 'your', 'coke', 'like', 'beverage', '.', ':', ')', '[SEP]']
['[CLS]', 'yum', '.', 'baja', 'fresh', 'is', 'the', 'place', 'to', 'go', 'if', 'you', 'want', 'something', 'a', 'step', 'up', 'from', 'chipotle', 'in', 'variety', 'and', 'warmth', 'as', 'far', 'as', 'the', 'food', 'goes',

In [9]:
tmp=np.array(new_attr)
if label_pred == 0:
  tmp = tmp * -1

for i in range(len(tmp)):
    if tmp[i]>0 or tmp[i]<0:
        tmp[i]=tmp[i]*100
        

In [10]:
# Visualize the attribution for the attribution
from captum.attr import visualization as viz

# indices = tokenized_sentence[0].detach().tolist()
# all_tokens = tokenizer.convert_ids_to_tokens(indices)

all_tokens = tokens
delta=torch.Tensor([0])
# confidence=np.max(curr_pred)
confidence = 1
ground_label=0
last_arg=0

print(confidence)

attributions_start_sum = attr[0]

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        tmp,
                        confidence, 
                        label_pred,
                        ground_label,
                        last_arg,
                        tmp.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])

1
 Visualizations For Start Position 


#Investigating BERT

In [ ]:
from transformers import TrainingArguments, Trainer
from bertviz import model_view
# Initialize our Trainer

from transformers import AutoModelForSequenceClassification,DistilBertConfig,BertConfig

model = AutoModelForSequenceClassification.from_pretrained("randellcotta/distilbert-base-uncased-finetuned-yelp-polarity",output_hidden_states=False, output_attentions=True )
test_output = model(tokenized_sentence)
attentions = test_output['attentions']
tokens = tokenizer.convert_ids_to_tokens(tokenized_sentence[0]) 
model_view(attentions, tokens)



In [13]:
import torch
# attentions_trch = torch.from_numpy(attentions[0])
print(len(attentions))

print(attentions[-1].shape)
# print(attentions[-1][:,0,:].shape)
# score = np.mean(attentions[0][:, 0, :], axis=1,keepdim=False)
print(torch.mean(attentions[-1], dim=1, keepdim=False).shape)
score_1 = torch.mean(attentions[-1], dim=1, keepdim=False)[:, 0, :] 
score_2 = torch.mean(torch.mean(attentions[-1], dim=1, keepdim=False), dim=1,keepdim=False)

# print(torch.mean(attentions[-1], dim=1, keepdim=False).shape)
print(score_1.shape)
print(score_2.shape)

# 6th (final) transformer block (right before classification), take average over 12 heads, take only 0th row

6
torch.Size([1, 12, 104, 104])
torch.Size([1, 104, 104])
torch.Size([1, 104])
torch.Size([1, 104])


In [ ]:
temp=torch.squeeze(score_1)
if label_pred == 0:
  temp = temp * -1

print(temp)
for i in range(len(temp)):
    if temp[i]>0 or temp[i]<0:
        temp[i]=temp[i]*100
        

tensor([0.0323, 0.0829, 0.0186, 0.0192, 0.0198, 0.0149, 0.0082, 0.0052, 0.0018,
        0.0109, 0.0058, 0.0020, 0.0048, 0.0028, 0.0019, 0.0028, 0.0129, 0.0088,
        0.0157, 0.0129, 0.0189, 0.0160, 0.0185, 0.0459, 0.0391, 0.0286, 0.0023,
        0.0014, 0.0017, 0.0022, 0.0026, 0.0026, 0.0043, 0.0033, 0.0022, 0.0039,
        0.0058, 0.0018, 0.0013, 0.0013, 0.0013, 0.0014, 0.0016, 0.0214, 0.0033,
        0.0016, 0.0010, 0.0014, 0.0081, 0.0042, 0.0072, 0.0053, 0.0197, 0.0118,
        0.0422, 0.0366, 0.0263, 0.0204, 0.0060, 0.0043, 0.0091, 0.0144, 0.0147,
        0.0117, 0.0136, 0.0163, 0.0769, 0.0245, 0.0027, 0.0112, 0.0017, 0.0022,
        0.0005, 0.0023, 0.0003, 0.0004, 0.0044, 0.0117, 0.0133, 0.0233, 0.0030,
        0.0035, 0.0046, 0.0208], grad_fn=<SqueezeBackward0>)


In [ ]:
# Visualize the attribution for the attribution
from captum.attr import visualization as viz

indices = tokenized_sentence[0].detach().tolist()
all_tokens = tokenizer.convert_ids_to_tokens(indices)
delta=torch.Tensor([0])
confidence=np.max(curr_pred)
label_pred=np.argmax(curr_pred)
ground_label=0
last_arg=0

print(confidence)

attributions_start_sum = attr[0]

#  Change first 3 zero to 1 if positive class

start_position_vis = viz.VisualizationDataRecord(
                        temp,
                        confidence, 
                        label_pred,
                        ground_label,
                        last_arg,
                        temp.sum(),       
                        all_tokens,
                        delta)

print('\033[1m', 'Visualizations For Start Position', '\033[0m')
viz.visualize_text([start_position_vis])

3.8966718
 Visualizations For Start Position 


In [14]:
!pip install bertviz

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
idx = 2022
print(df.text[idx])
print(df.label[idx])
sentence_2 = df.text[idx]
attr_2 = attention_sst_sst[idx,:]
tokenized_sentence_2 = tokenized_sentence=tokenizer.encode(sentence,add_special_tokens=False,padding=True,truncation=True,return_tensors='pt')
curr_pred_2 = preds[idx]

# print(attr)
print(curr_pred_2)
print(np.mean(attr_2))

Great sushi.  I've been coming here for the past 5 years.  Their service can be a little slow at times, especially weekend nights.  However the rolls & sushimi are top notch.
1
[-4.3504233  3.8966718]
0.001953125


In [ ]:
# Our actual model
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 